In [10]:
# marine_debris_yolov5_test.py

import os
import glob
import yaml
import torch
import cv2
from collections import Counter
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# -----------------------
# Hyperparameters
# -----------------------

DATA_YAML = "data.yaml"
WEIGHTS = "yolov5/runs/train/exp11/weights/best.pt"

IMG_SIZE = 640
CONF_THRES = 0.25
BATCH_SIZE = 8

SAVE_DIR = "runs/test_results"
os.makedirs(SAVE_DIR, exist_ok=True)

# -----------------------
# Read data.yaml
# -----------------------

with open(DATA_YAML) as f:
    data = yaml.safe_load(f)

dataset_root = data["path"]
test_folder = data["test"]

TEST_DIR = os.path.join(dataset_root, test_folder)

print("Test folder:", TEST_DIR)

# -----------------------
# Collect images
# -----------------------

img_paths = []

for ext in ["*.jpg","*.png","*.jpeg","*.bmp"]:
    img_paths += glob.glob(os.path.join(TEST_DIR, ext))

img_paths = sorted(img_paths)

print("Test images found:", len(img_paths))

if len(img_paths) == 0:
    raise RuntimeError("No test images found")

# -----------------------
# Load model
# -----------------------

device = "cuda" if torch.cuda.is_available() else "cpu"

model = torch.hub.load(
    "ultralytics/yolov5",
    "custom",
    path=WEIGHTS,
    force_reload=True
)

model.to(device)

model.conf = CONF_THRES

print("Model loaded on", device)

# -----------------------
# Batched inference
# -----------------------

class_counter = Counter()
total_det = 0

for i in range(0, len(img_paths), BATCH_SIZE):

    batch_paths = img_paths[i:i+BATCH_SIZE]

    results = model(batch_paths, size=IMG_SIZE)

    for j, det in enumerate(results.xyxy):

        img_path = batch_paths[j]
        img = cv2.imread(img_path)

        for *xyxy, conf, cls in det.cpu().numpy():

            x1,y1,x2,y2 = map(int, xyxy)

            cls = int(cls)
            name = model.names[cls]

            label = f"{name} {conf:.2f}"

            cv2.rectangle(img,(x1,y1),(x2,y2),(0,0,255),2)

            cv2.putText(
                img,
                label,
                (x1,y1-5),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0,0,255),
                1
            )

            class_counter[name]+=1
            total_det+=1

        out_path = os.path.join(SAVE_DIR, os.path.basename(img_path))
        cv2.imwrite(out_path,img)

    torch.cuda.empty_cache()

print("\nTotal detections:",total_det)

print("\nDetections per class:")
for k,v in class_counter.items():
    print(k,":",v)

# -----------------------
# Show example
# -----------------------

example = glob.glob(SAVE_DIR+"/*")[0]

img = cv2.imread(example)
img = cv2.cvtColor(img,cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10,8))
plt.imshow(img)
plt.axis("off")
plt.title("Example Detection")
plt.show()

Test folder: data\images/test
Test images found: 2293
Downloading: "https://github.com/ultralytics/yolov5/zipball/master" to C:\Users\nisan/.cache\torch\hub\master.zip


YOLOv5  2026-3-15 Python-3.10.11 torch-2.10.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)

Fusing layers... 
Model summary: 157 layers, 7042489 parameters, 0 gradients, 15.9 GFLOPs
Adding AutoShape... 


Model loaded on cuda

Total detections: 495

Detections per class:
class9 : 119
class8 : 111
class6 : 109
class2 : 150
class1 : 1
class11 : 4
class5 : 1
